# Random Forest — Riesgo de incumplimiento de SLA (F1-06)

**Objetivo:** entrenar un Random Forest sobre el mismo `train.parquet`/`test.parquet` del baseline (`02_modelo_baseline.ipynb`), para evaluar si un modelo no lineal captura interacciones entre variables (por ejemplo, zona × franja horaria) que la regresión logística, al combinar variables con pesos fijos, no puede representar.

**Por qué probar esto:** la regresión logística asigna un peso fijo a cada variable, sin importar el valor de las demás. Un árbol de decisión, en cambio, encadena preguntas ("¿franja es noche? → ¿zona de riesgo alto? → ¿volumen > X?"), por lo que el efecto de una variable puede depender de las otras. Si esa clase de interacción existe en los datos, un árbol debería mejorar especialmente el recall de `Low`, que es donde el baseline falla.

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

PROCESSED = "../../data/processed"

train_df = pd.read_parquet(f"{PROCESSED}/train.parquet")
test_df = pd.read_parquet(f"{PROCESSED}/test.parquet")
print("Train:", train_df.shape, "| Test:", test_df.shape)

Train: (719870, 27) | Test: (178545, 27)


## 1. Mismos features, mismo tratamiento de nulos

Reutilizamos exactamente la selección de features, la lógica de nulos y el one-hot encoding del baseline (`02_modelo_baseline.ipynb`) — cambiar de modelo no cambia esas decisiones, que dependen de los datos, no del algoritmo.

In [2]:
feature_cols_num = [
    "n_packages", "total_volume_cm3", "total_planned_service_seconds",
    "window_duration_min", "volumen_promedio_paquete_cm3",
    "paradas_por_ruta", "paquetes_por_ruta", "distancia_a_siguiente_km",
    "zona_riesgo_low",
]
feature_cols_bool = ["has_time_window", "any_rejected", "any_attempted"]
feature_cols_cat = ["station_code", "franja_horaria"]


def prep(df, median_dist=None):
    df = df.copy()
    df["window_duration_min"] = df["window_duration_min"].fillna(0)
    df["franja_horaria"] = df["franja_horaria"].cat.add_categories("sin_ventana").fillna("sin_ventana")
    if median_dist is None:
        median_dist = df["distancia_a_siguiente_km"].median()
    df["distancia_a_siguiente_km"] = df["distancia_a_siguiente_km"].fillna(median_dist)
    for c in feature_cols_bool:
        df[c] = df[c].astype(int)
    return df, median_dist


train_df, median_dist = prep(train_df)
test_df, _ = prep(test_df, median_dist=median_dist)

all_feats = feature_cols_num + feature_cols_bool + feature_cols_cat
print("Nulos restantes en train (debe ser 0):", train_df[all_feats].isna().sum().sum())
print("Nulos restantes en test (debe ser 0):", test_df[all_feats].isna().sum().sum())

Nulos restantes en train (debe ser 0): 0
Nulos restantes en test (debe ser 0): 0


In [3]:
X_train = pd.get_dummies(train_df[all_feats], columns=feature_cols_cat)
X_test = pd.get_dummies(test_df[all_feats], columns=feature_cols_cat)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_df["route_score"]
y_test = test_df["route_score"]

print("X_train:", X_train.shape, "| X_test:", X_test.shape)

X_train: (719870, 34) | X_test: (178545, 34)


## 2. Qué cambia (y qué no) respecto al preprocesamiento del baseline

**Se elimina el `StandardScaler`.** La regresión logística lo necesitaba porque combina variables con una suma de pesos, y una variable con magnitudes grandes (`total_volume_cm3`) dominaba esa suma solo por su escala. Un árbol no suma nada: en cada nodo pregunta "¿este valor es mayor a X?", y esa pregunta encuentra el mismo punto de corte útil sin importar en qué unidad esté medida la variable. Escalar no cambia el resultado de un árbol, así que no aporta nada acá.

**Se mantiene el one-hot encoding.** Esto no es para resolver un problema de escala — es para poder representarle categorías de texto (`station_code`, `franja_horaria`) al modelo en formato numérico, algo que Random Forest en scikit-learn también necesita.

**El desbalance de clases (1.7% `Low`) sigue siendo un problema, sin importar el modelo.** No es un problema de la regresión logística en particular — es un problema de cómo están repartidos los datos. Cualquier modelo que minimice su error total puede lograrlo casi ignorando a `Low`, porque es solo 1.7% de los casos. Por eso usamos `class_weight` también acá.

## 3. Entrenamiento

Usamos `class_weight="balanced_subsample"` en vez de `"balanced"`: Random Forest entrena cada árbol sobre una muestra al azar (bootstrap) distinta de los datos, y `"balanced_subsample"` recalcula el peso de cada clase **dentro de cada muestra individual**, en vez de una sola vez sobre todo el train. Es la versión de `class_weight` pensada específicamente para un ensamble por bagging como este.

`max_depth=12` limita cuántas preguntas encadenadas puede hacer cada árbol. Es una decisión deliberada, no arbitraria — ver sección 5 para la evidencia de por qué no se dejó sin límite.

In [4]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
print("Modelo entrenado.")

Modelo entrenado.


## 4. Evaluación: Random Forest vs. baseline

Sumamos ROC-AUC (promedio one-vs-rest para las 3 clases) a las métricas del baseline. Un ROC-AUC de 0.5 equivale a un modelo que ordena las probabilidades al azar; 1.0 es un ordenamiento perfecto.

In [5]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

print(classification_report(y_test, y_pred))
print("Matriz de confusión (filas=real, columnas=predicho), orden Low/Medium/High:")
print(confusion_matrix(y_test, y_pred, labels=["Low", "Medium", "High"]))

auc = roc_auc_score(y_test, y_proba, multi_class="ovr", labels=model.classes_)
print("ROC-AUC (ovr, macro):", round(auc, 4))

              precision    recall  f1-score   support

        High       0.57      0.46      0.51     74901
         Low       0.02      0.18      0.03      3047
      Medium       0.66      0.58      0.62    100597

    accuracy                           0.52    178545
   macro avg       0.42      0.41      0.39    178545
weighted avg       0.61      0.52      0.56    178545

Matriz de confusión (filas=real, columnas=predicho), orden Low/Medium/High:
[[  549  1560   938]
 [16955 58478 25164]
 [12419 28074 34408]]
ROC-AUC (ovr, macro): 0.6033


**Comparación con el baseline (regresión logística, `02_modelo_baseline.ipynb`) — ambos con `zona_riesgo_low` ya corregida (suavizado bayesiano):**

| Métrica | Regresión logística | Random Forest |
|---|---|---|
| `Low` — precision / recall / f1 | 0.02 / 0.18 / 0.04 | 0.02 / 0.18 / 0.03 |
| `Medium` — precision / recall / f1 | 0.68 / 0.57 / 0.62 | 0.66 / 0.58 / 0.62 |
| `High` — precision / recall / f1 | 0.55 / 0.50 / 0.52 | 0.57 / 0.45 / 0.50 |
| Accuracy | 0.53 | 0.52 |
| ROC-AUC (ovr, macro) | 0.6102 | 0.6029 |

El recall de `Low` es prácticamente idéntico entre los dos modelos (18%), y el ROC-AUC de la regresión logística es levemente **mejor** que el de Random Forest, no peor. Cambiar a un modelo no lineal no mejoró nada en esta ronda.

## 5. ¿Por qué no simplemente aumentar la profundidad o la cantidad de árboles?

Hipótesis inicial: si un árbol más profundo puede encadenar más preguntas, ¿no debería capturar mejor las interacciones y mejorar el recall de `Low`? Se probó entrenando un Random Forest **sin límite de profundidad** y comparando su desempeño en train contra test — no solo mirar el resultado final, sino comprobar si el modelo está aprendiendo un patrón real o memorizando.

In [6]:
model_deep = RandomForestClassifier(
    n_estimators=50,
    max_depth=None,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)
model_deep.fit(X_train, y_train)

pred_train_deep = model_deep.predict(X_train)
pred_test_deep = model_deep.predict(X_test)

print("=== TRAIN (profundidad sin límite) ===")
print(classification_report(y_train, pred_train_deep))
print("=== TEST (profundidad sin límite) ===")
print(classification_report(y_test, pred_test_deep))

=== TRAIN (profundidad sin límite) ===
              precision    recall  f1-score   support

        High       1.00      1.00      1.00    304543
         Low       1.00      1.00      1.00     12279
      Medium       1.00      1.00      1.00    403048

    accuracy                           1.00    719870
   macro avg       1.00      1.00      1.00    719870
weighted avg       1.00      1.00      1.00    719870

=== TEST (profundidad sin límite) ===
              precision    recall  f1-score   support

        High       0.53      0.47      0.50     74901
         Low       0.00      0.00      0.00      3047
      Medium       0.63      0.69      0.66    100597

    accuracy                           0.59    178545
   macro avg       0.38      0.39      0.39    178545
weighted avg       0.57      0.59      0.58    178545



**Resultado:** en train, el modelo llega a 1.00 de precision/recall/f1 en las tres clases — memorización perfecta, incluida `Low`. En test, el recall de `Low` cae drásticamente (muy por debajo del modelo con `max_depth=12`): peor que el modelo con profundidad limitada y peor que el baseline.

Esto confirma que la hipótesis ("más profundidad = más conexiones = mejor") es incorrecta en la práctica: dejar que el árbol haga preguntas ilimitadas no le permite generalizar mejor — le permite memorizar las filas `Low` de train con reglas tan específicas que no aplican a ninguna ruta `Low` nueva. Es el mismo problema conceptual que descartar `route_id`/`stop_id` como feature (evitar que el modelo memorice identificadores en vez de aprender patrones), aplicado ahora a la profundidad del árbol en lugar de a una columna.

Por eso `max_depth=12` (sección 3) no es un valor arbitrario: es la profundidad que evita este colapso, aunque tampoco resuelve el problema de `Low`.

## 6. Qué variables usa el modelo

Con el modelo de `max_depth=12` (el que se usaría en la práctica), miramos qué features concentran la capacidad predictiva.

In [7]:
importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Top 10 features por importancia:")
print(importances.head(10))

Top 10 features por importancia:
zona_riesgo_low             0.611456
paradas_por_ruta            0.122077
paquetes_por_ruta           0.109522
station_code_DLA9           0.021416
station_code_DLA8           0.012189
station_code_DCH1           0.009782
station_code_DCH4           0.009677
distancia_a_siguiente_km    0.008507
station_code_DBO3           0.007917
station_code_DLA7           0.007870
dtype: float64


`zona_riesgo_low` sigue concentrando la mayor parte de la importancia (~61%), seguida de `paradas_por_ruta` y `paquetes_por_ruta` (~12% y ~11%). El resto de los features (franja horaria, distancia, estación) aporta relativamente poco. Esto es consistente con el hallazgo de una prueba adicional (ver README): un modelo entrenado **solo con `zona_riesgo_low`** obtiene casi el mismo recall de `Low` (~17%) que el modelo completo con las 14 features — señal de que la señal útil está concentrada ahí, y las demás variables de F1-04 no están aportando interacciones adicionales relevantes.

## 7. Conclusión de negocio

**En una línea:** cambiar de un modelo lineal a Random Forest no mejora la detección de rutas `Low` (18% de recall en ambos, ROC-AUC incluso levemente peor en Random Forest) — la complejidad adicional no está pagando, y la causa más probable es que casi toda la señal útil ya está concentrada en `zona_riesgo_low`, sin interacciones adicionales fuertes que un árbol pueda explotar con las features actuales.

**Para el negocio:** hoy ningún modelo de los dos que probamos es confiable para decidir qué rutas marcar como de riesgo — ambos dejan pasar la enorme mayoría de las rutas `Low` reales. No se recomienda todavía usar esto como herramienta operativa.

**Pendiente / próximo paso real:** en vez de seguir iterando el tipo de modelo, el cuello de botella parece ser de **señal disponible**, no de algoritmo — las features de F1-04 (densidad de paquetes, distancia, ventana horaria) no aportan nada por encima de `zona_riesgo_low` sola. Haría falta investigar señales cualitativamente nuevas (por ejemplo, historial de la estación, comportamiento temporal más fino), o aceptar que con solo 102 rutas `Low` en todo el dataset, hay un techo real de cuánto puede aprender cualquier modelo sobre esta clase, independientemente del feature engineering.